# Comparaison simple : facteurs historiques et composites améliorés

Ce notebook compare, séparément pour **STOXX EUROPE 600** et **EUROPE SMALL CAP** :

1. le facteur historique du screen ;
2. un composite amélioré qui combine ce facteur historique avec une ou plusieurs nouvelles variables ;
3. l'effet incrémental de chaque nouvelle variable ou dimension change ajoutée seule au facteur historique ;
4. les performances Top, les ratios relatifs et les figures Plotly.

Les cellules `COMPOSITE_CONFIGS_STOXX` et `COMPOSITE_CONFIGS_SMALL` sont les seules cellules de paramétrage à modifier en priorité. Les poids sont relatifs : ils peuvent être changés manuellement sans modifier le pipeline. Les commentaires, messages et cette documentation restent volontairement en français pour conserver la convention du projet.

Les résultats sont écrits dans `exports/factor_core_upgrade_comparison_STOXX600` et `exports/factor_core_upgrade_comparison_SMALL`. Le notebook est livré sans sorties d'exécution.

In [ ]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

from factor_config import LOWER_IS_BETTER, signal_options
from func import (
    calculate_benchmark_performance,
    calculate_performance_ratios,
    combine_backtest_performances,
    export_backtest_results,
    load_backtest_data,
    plot_performance_comparison,
    test_composite_signals,
)

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'func.py').exists():
    raise RuntimeError(
        'Définissez le répertoire de travail Jupyter sur C:\\dev\\factor_backtest.'
    )

DATA_DIR = REPO_ROOT / 'data'
SCREEN_PATH = DATA_DIR / 'screen_aggregate.parquet'
RETURNS_PATH = DATA_DIR / 'returns.parquet'
EXPORT_ROOT = REPO_ROOT / 'exports'
LIST_NOIRE_PATH = None
START_DATE = '2007-12-01'
PERCENTILE = 0.13
N_JOBS = 1
PERIOD_BREAKPOINTS = [2009, 2013, 2017, 2020, 2022, 2024, 2026]

METRIC_COLUMNS = [
    'active_cagr',
    'top_worst_cagr',
    'top_information_ratio',
    'robust_score',
    'active_max_drawdown',
    'tracking_error_annualized',
    'min_rolling_3y_cagr',
    'top_bench_ratio',
    'top_worst_ratio',
    'top_annualized_return',
    'top_annualized_volatility',
    'top_max_drawdown',
    'observation_count',
    'years',
]
STABILITY_METRIC_COLUMNS = [
    'path_trend_r2',
    'path_log_slope_annualized',
    'path_residual_vol_annualized',
    'relative_ulcer_index',
    'relative_max_drawdown',
    'max_underwater_months',
    'underwater_events',
    'recovered_events',
    'recovery_rate',
    'median_recovery_months',
    'max_recovery_months',
    'positive_12m_active_rate',
    'positive_36m_active_rate',
    'monthly_active_hit_rate',
    'active_tail_5pct',
    'months',
]
STABILITY_SCORE_COLUMNS = [
    'relative_smoothness_score',
    'relative_persistence_score',
]


def safe_name(value):
    """Produit un identifiant court pour les fichiers et les noms de groupes."""
    return re.sub(r'[^A-Za-z0-9._-]+', '_', str(value)).strip('._') or 'variable'


def resolve_baseline_columns(candidates, available_columns):
    """Choisit le premier ancien facteur réellement présent dans le screen."""
    resolved = {}
    for family, options in candidates.items():
        selected = next((name for name in options if name in available_columns), None)
        if selected is None:
            raise KeyError(
                f'Aucun facteur historique disponible pour {family}: {options}'
            )
        resolved[family] = selected
    return resolved


def make_signal_config(specs):
    """Transforme une liste lisible de composantes en configuration du moteur."""
    config = {}
    for spec in specs:
        variable = spec['variable']
        dimension = spec['dimension']
        higher_is_better = bool(spec.get('higher_is_better', variable not in LOWER_IS_BETTER))
        expected_direction = variable not in LOWER_IS_BETTER
        if higher_is_better != expected_direction:
            raise ValueError(
                f'Direction incohérente pour {variable}: utilisez higher_is_better={expected_direction}.'
            )
        options = config.setdefault(
            variable,
            signal_options(higher_is_better=higher_is_better),
        )
        options[f'weight_{dimension}'] = float(spec.get('weight', 1.0))
    return config


def make_baseline_config(variable):
    """Construit le facteur historique seul, au niveau du screen."""
    return {variable: signal_options(level=1.0, higher_is_better=True)}


def make_incremental_candidate_config(baseline_variable, spec):
    """Ajoute une composante, y compris une autre dimension du même facteur historique."""
    baseline_spec = {
        'variable': baseline_variable,
        'dimension': 'level',
        'weight': 1.0,
        'higher_is_better': True,
    }
    return make_signal_config([baseline_spec, spec])


def collect_variables(baseline_columns, composite_specs):
    """Retourne les colonnes nécessaires sans doublons."""
    variables = list(baseline_columns.values())
    variables.extend(
        spec['variable']
        for specs in composite_specs.values()
        for spec in specs
    )
    return list(dict.fromkeys(variables))


def comparison_table(metrics, old_path, upgrade_path, family, upgrade_label='upgrade', metric_columns=None):
    """Compare les métriques historiques et améliorées période par période."""
    keys = [column for column in ('scope', 'period_id', 'period_label') if column in metrics.columns]
    requested_columns = METRIC_COLUMNS if metric_columns is None else metric_columns
    columns = [column for column in requested_columns if column in metrics.columns]
    old = metrics.loc[metrics['test_path'].eq(old_path), keys + columns].copy()
    upgrade = metrics.loc[metrics['test_path'].eq(upgrade_path), keys + columns].copy()
    if old.empty or upgrade.empty:
        raise KeyError(f'Métriques introuvables pour {family}: {old_path} / {upgrade_path}')
    old = old.rename(columns={column: f'{column}_old' for column in columns})
    upgrade = upgrade.rename(columns={column: f'{column}_{upgrade_label}' for column in columns})
    joined = upgrade.merge(old, on=keys, how='outer')
    joined.insert(0, 'family', family)
    for column in columns:
        joined[f'delta_{column}'] = joined[f'{column}_{upgrade_label}'] - joined[f'{column}_old']
    required = {'active_cagr', 'top_worst_cagr', 'top_information_ratio'}
    if required.issubset(columns):
        joined['upgrade_gate'] = (
            joined[f'active_cagr_{upgrade_label}'].gt(0)
            & joined[f'top_worst_cagr_{upgrade_label}'].gt(0)
            & joined[f'top_information_ratio_{upgrade_label}'].gt(0)
        )
        joined['old_gate'] = (
            joined['active_cagr_old'].gt(0)
            & joined['top_worst_cagr_old'].gt(0)
            & joined['top_information_ratio_old'].gt(0)
        )
        joined['performance_improved'] = (
            joined[f'delta_active_cagr'].gt(0)
            & joined[f'delta_top_worst_cagr'].gt(0)
            & joined[f'delta_top_information_ratio'].gt(0)
        )
    if {'active_max_drawdown', 'tracking_error_annualized'}.issubset(columns):
        joined['relative_risk_not_worse'] = (
            joined['delta_active_max_drawdown'].le(0)
            & joined['delta_tracking_error_annualized'].le(0)
        )
    if {'top_annualized_volatility', 'top_max_drawdown'}.issubset(columns):
        joined['absolute_risk_not_worse'] = (
            joined['delta_top_annualized_volatility'].le(0)
            & joined['delta_top_max_drawdown'].le(0)
        )
    sort_columns = [column for column in ('period_id', 'scope') if column in joined.columns]
    return joined.sort_values(sort_columns).reset_index(drop=True)


def calculate_relative_path_stability(performance, portfolio='Portfolio', benchmark='Benchmark'):
    """Mesure la régularité de la courbe Top/Benchmark."""
    empty = {column: np.nan for column in STABILITY_METRIC_COLUMNS}
    empty['months'] = 0
    required = [portfolio, benchmark]
    if not isinstance(performance, pd.DataFrame) or not set(required).issubset(performance.columns):
        return empty
    frame = performance.loc[:, required].apply(pd.to_numeric, errors='coerce').dropna()
    if frame.empty:
        return empty
    monthly = frame.resample('ME').last().dropna()
    if len(monthly) < 2:
        empty['months'] = len(monthly)
        return empty
    relative = (monthly[portfolio] / monthly[benchmark]).replace([np.inf, -np.inf], np.nan).dropna()
    if len(relative) < 2 or (relative <= 0).any():
        empty['months'] = len(relative)
        return empty
    log_relative = np.log(relative / relative.iloc[0])
    x = np.arange(len(log_relative), dtype=float)
    slope, intercept = np.polyfit(x, log_relative.to_numpy(), 1)
    fitted = slope * x + intercept
    residual = log_relative.to_numpy() - fitted
    ss_total = float(np.square(log_relative.to_numpy() - log_relative.mean()).sum())
    ss_residual = float(np.square(residual).sum())
    trend_r2 = 1.0 if ss_total == 0 else max(0.0, 1.0 - ss_residual / ss_total)
    drawdown = relative / relative.cummax() - 1.0
    underwater = drawdown.lt(0)
    runs = underwater.astype(int).groupby(underwater.ne(underwater.shift()).cumsum()).sum()
    underwater_groups = underwater.ne(underwater.shift()).cumsum()
    recovery_durations = []
    underwater_events = 0
    for _, segment in underwater.groupby(underwater_groups):
        if not bool(segment.iloc[0]):
            continue
        underwater_events += 1
        if segment.index[-1] != underwater.index[-1]:
            recovery_durations.append(int(segment.sum()))
    active_returns = monthly[portfolio].pct_change() - monthly[benchmark].pct_change()
    rolling_12m = relative.pct_change(12).dropna()
    rolling_36m = relative.pct_change(36).dropna()
    return {
        'path_trend_r2': trend_r2,
        'path_log_slope_annualized': float(slope * 12.0),
        'path_residual_vol_annualized': float(np.std(residual, ddof=1) * np.sqrt(12)) if len(residual) > 1 else np.nan,
        'relative_ulcer_index': float(np.sqrt(np.square(drawdown).mean())),
        'relative_max_drawdown': float(-drawdown.min()),
        'max_underwater_months': int(runs.max()) if not runs.empty else 0,
        'underwater_events': int(underwater_events),
        'recovered_events': int(len(recovery_durations)),
        'recovery_rate': (float(len(recovery_durations) / underwater_events) if underwater_events else np.nan),
        'median_recovery_months': (float(np.median(recovery_durations)) if recovery_durations else np.nan),
        'max_recovery_months': (int(max(recovery_durations)) if recovery_durations else np.nan),
        'positive_12m_active_rate': float((rolling_12m > 0).mean()) if not rolling_12m.empty else np.nan,
        'positive_36m_active_rate': float((rolling_36m > 0).mean()) if not rolling_36m.empty else np.nan,
        'monthly_active_hit_rate': float((active_returns.dropna() > 0).mean()) if active_returns.notna().any() else np.nan,
        'active_tail_5pct': float(active_returns.dropna().quantile(0.05)) if active_returns.notna().any() else np.nan,
        'months': int(len(relative)),
    }

def calculate_relative_stability_scores(path_metrics, persistence_metrics):
    """Réduit les mesures détaillées à deux scores lisibles sur 0-100."""
    def inverse(value, scale):
        if pd.isna(value):
            return np.nan
        return 1.0 / (1.0 + max(float(value), 0.0) / scale)

    smoothness_parts = [
        path_metrics.get('path_trend_r2'),
        inverse(path_metrics.get('path_residual_vol_annualized'), 0.15),
        inverse(path_metrics.get('relative_ulcer_index'), 0.10),
        inverse(path_metrics.get('median_recovery_months'), 12.0) if pd.notna(path_metrics.get('median_recovery_months')) else 1.0,
    ]
    persistence_parts = [
        path_metrics.get('positive_12m_active_rate'),
        path_metrics.get('positive_36m_active_rate'),
        path_metrics.get('monthly_active_hit_rate'),
        persistence_metrics.get('strict_positive_period_rate'),
    ]
    smoothness = pd.Series(smoothness_parts, dtype='float64').dropna()
    persistence = pd.Series(persistence_parts, dtype='float64').dropna()
    return {
        'relative_smoothness_score': float(100.0 * smoothness.mean()) if not smoothness.empty else np.nan,
        'relative_persistence_score': float(100.0 * persistence.mean()) if not persistence.empty else np.nan,
    }

def _compact_display(table, columns):
    """Garde seulement les colonnes utiles pour l'affichage interactif."""
    if table is None or table.empty:
        return table
    selected = [column for column in columns if column in table.columns]
    return table.loc[:, selected].copy()


def display_key_comparison(table):
    """Affiche le rendement relatif et les principaux contrôles de risque."""
    return _compact_display(table, [
        'family', 'scope', 'period_id',
        'delta_active_cagr', 'delta_top_information_ratio', 'delta_top_worst_cagr',
        'relative_risk_not_worse', 'absolute_risk_not_worse', 'performance_improved',
    ])


def display_key_stability(table):
    """Affiche les deux scores de stabilité relative, ancien contre upgrade."""
    return _compact_display(table, [
        'family', 'scope', 'period_id',
        'relative_smoothness_score_old', 'relative_smoothness_score_upgrade',
        'delta_relative_smoothness_score',
        'relative_persistence_score_old', 'relative_persistence_score_upgrade',
        'delta_relative_persistence_score',
    ])


def display_key_incremental(table):
    """Affiche l'effet marginal de chaque composante ajoutée."""
    return _compact_display(table, [
        'family', 'variable', 'dimension', 'scope', 'period_id',
        'delta_active_cagr', 'delta_top_information_ratio', 'delta_top_worst_cagr',
        'delta_relative_smoothness_score', 'delta_relative_persistence_score',
        'relative_risk_not_worse', 'absolute_risk_not_worse', 'performance_improved',
    ])

def run_market(market, benchmark, output_name, baseline_candidates, composite_specs):
    """Exécute un marché : composites, incrémentaux, métriques et manifest."""
    available_columns = set(pq.ParquetFile(SCREEN_PATH).schema_arrow.names)
    baseline_columns = resolve_baseline_columns(baseline_candidates, available_columns)
    load_variables = collect_variables(baseline_columns, composite_specs)
    missing = [column for column in load_variables if column not in available_columns]
    if missing:
        raise KeyError(f'Variables absentes du screen pour {market}: {missing}')

    screen, returns = load_backtest_data(
        screen_path=SCREEN_PATH,
        returns_path=RETURNS_PATH,
        variables=load_variables,
        bench=benchmark,
        start_date=START_DATE,
        lookback_periods=12,
        compact_dtypes=True,
    )
    screen['Date'] = pd.to_datetime(screen['Date'])
    if f'Weight in {benchmark}' not in screen.columns:
        raise KeyError(f'La colonne Weight in {benchmark} est absente du screen.')
    benchmark_performance = calculate_benchmark_performance(
        screen=screen, returns=returns, bench=benchmark, start_date=START_DATE
    )
    run_options = {
        'bench': benchmark,
        'bench_perf': benchmark_performance,
        'percentile': PERCENTILE,
        'start_date': START_DATE,
        'freq_rebal': 1,
        'fill_method': 'copy',
        'n_jobs': N_JOBS,
        'retain_builders': False,
        'monthly_base_cache': {},
        'period_breakpoints': PERIOD_BREAKPOINTS,
        'show_plot': False,
        'build_figure': False,
    }

    composite_configs = {}
    for family, specs in composite_specs.items():
        for spec in specs:
            if spec['variable'] not in screen.columns:
                raise KeyError(f'{spec["variable"]} absent après chargement pour {market}.')
        composite_configs[f'old__{family}'] = make_baseline_config(baseline_columns[family])
        composite_configs[f'upgrade__{family}'] = make_signal_config(specs)

    composite_batch = test_composite_signals(
        screen=screen,
        returns=returns,
        composite_configs=composite_configs,
        list_noire_path=LIST_NOIRE_PATH,
        score_prefix=f'Score_CoreUpgrade_{safe_name(market)}',
        **run_options,
    )

    incremental_specs = {}
    incremental_configs = {}
    for family, specs in composite_specs.items():
        components = [
            spec for spec in specs
            if not (spec['variable'] == baseline_columns[family] and spec['dimension'] == 'level')
        ]
        for index, spec in enumerate(components, start=1):
            # Les identifiants courts évitent les chemins Windows trop longs ;
            # la variable et la dimension complètes restent dans les tables exportées.
            group_key = f'{family}__{index}'
            incremental_specs[group_key] = {'family': family, **spec}
            old_name = f'inc_old_{family}_{index}'
            candidate_name = f'inc_candidate_{family}_{index}'
            incremental_configs[old_name] = make_baseline_config(baseline_columns[family])
            incremental_configs[candidate_name] = make_incremental_candidate_config(
                baseline_columns[family], spec
            )
            incremental_specs[group_key]['old_name'] = old_name
            incremental_specs[group_key]['candidate_name'] = candidate_name

    if incremental_configs:
        incremental_batch = test_composite_signals(
            screen=composite_batch['screen'],
            returns=returns,
            composite_configs=incremental_configs,
            list_noire_path=LIST_NOIRE_PATH,
            score_prefix=f'Score_Incremental_{safe_name(market)}',
            **run_options,
        )
    else:
        incremental_batch = {'screen': composite_batch['screen'], 'results': {}}

    all_results = {'composites': composite_batch, 'incremental': incremental_batch}
    exported = export_backtest_results(
        results=all_results,
        output_dir=EXPORT_ROOT,
        export_name=output_name,
        export_html=False,
        export_png=False,
        export_holdings=False,
    )
    export_dir = Path(exported['export_dir'])
    metrics = exported['metrics'].copy()
    registry = exported['registry']
    path_by_name = {
        entry.get('metadata', {}).get('test_name'): entry.get('test_path')
        for entry in registry
        if entry.get('metadata', {}).get('test_name') and entry.get('test_path')
    }

    family_tables = []
    for family in composite_specs:
        table = comparison_table(
            metrics,
            path_by_name[f'old__{family}'],
            path_by_name[f'upgrade__{family}'],
            family,
        )
        family_tables.append(table)
    family_comparison = pd.concat(family_tables, ignore_index=True)
    family_comparison.to_csv(export_dir / 'family_composite_vs_old.csv', index=False)
    family_comparison.loc[family_comparison['period_id'].eq('total')].to_csv(
        export_dir / 'family_composite_vs_old_total.csv', index=False
    )

    # La stabilité recherchée porte sur la courbe relative Top/Benchmark.
    stability_names = []
    for family in composite_specs:
        stability_names.extend([f'old__{family}', f'upgrade__{family}'])
    for spec in incremental_specs.values():
        stability_names.extend([spec['old_name'], spec['candidate_name']])
    stability_names = list(dict.fromkeys(stability_names))
    template_path = path_by_name[f'old__{next(iter(composite_specs))}']
    period_template = metrics.loc[metrics['test_path'].eq(template_path), [
        'scope', 'period_id', 'period_label', 'actual_start_date', 'actual_end_date'
    ]].drop_duplicates(['scope', 'period_id']).copy()
    stability_rows = []
    persistence_rows = []
    for test_name in stability_names:
        test_path = path_by_name[test_name]
        performance = combine_backtest_performances(
            results=all_results,
            selections={
                'Portfolio': (test_path, 'Top'),
                'Benchmark': (test_path, 'Bench'),
            },
        )
        period_rows = metrics.loc[metrics['test_path'].eq(test_path)]
        subperiod_rows = period_rows.loc[period_rows['scope'].eq('subperiod')]
        persistence_rows.append({
            'test_name': test_name,
            'test_path': test_path,
            'subperiod_count': int(len(subperiod_rows)),
            'active_positive_period_rate': float(subperiod_rows['active_cagr'].gt(0).mean()) if len(subperiod_rows) else np.nan,
            'ir_positive_period_rate': float(subperiod_rows['top_information_ratio'].gt(0).mean()) if len(subperiod_rows) else np.nan,
            'worst_positive_period_rate': float(subperiod_rows['top_worst_cagr'].gt(0).mean()) if len(subperiod_rows) else np.nan,
            'strict_positive_period_rate': float((
                subperiod_rows['active_cagr'].gt(0)
                & subperiod_rows['top_information_ratio'].gt(0)
                & subperiod_rows['top_worst_cagr'].gt(0)
            ).mean()) if len(subperiod_rows) else np.nan,
        })
        for period in period_template.to_dict('records'):
            period_id = period['period_id']
            window = performance
            if str(period_id) != 'total':
                start = pd.to_datetime(period['actual_start_date'], errors='coerce')
                end = pd.to_datetime(period['actual_end_date'], errors='coerce')
                window = performance.loc[start:end] if pd.notna(start) and pd.notna(end) else performance.iloc[0:0]
            row = calculate_relative_path_stability(window)
            row.update({
                'test_name': test_name,
                'test_path': test_path,
                'scope': period['scope'],
                'period_id': period_id,
                'period_label': period['period_label'],
            })
            stability_rows.append(row)
    relative_path_stability = pd.DataFrame(stability_rows)
    persistence_summary = pd.DataFrame(persistence_rows)
    persistence_by_test = (
        persistence_summary.set_index('test_name').to_dict(orient='index')
        if not persistence_summary.empty else {}
    )
    score_rows = []
    for _, path_metrics in relative_path_stability.iterrows():
        persistence_metrics = persistence_by_test.get(path_metrics['test_name'], {})
        score_rows.append({
            **path_metrics.to_dict(),
            **calculate_relative_stability_scores(path_metrics, persistence_metrics),
        })
    relative_stability_scores = pd.DataFrame(score_rows)
    score_export_columns = [
        'test_name', 'test_path', 'scope', 'period_id', 'period_label',
        *STABILITY_SCORE_COLUMNS, 'months',
    ]
    relative_stability_scores = relative_stability_scores.loc[:, [
        column for column in score_export_columns if column in relative_stability_scores.columns
    ]]
    relative_stability_scores.to_csv(export_dir / 'relative_stability_scores.csv', index=False)

    family_stability_tables = []
    for family in composite_specs:
        family_stability_tables.append(comparison_table(
            relative_stability_scores,
            path_by_name[f'old__{family}'],
            path_by_name[f'upgrade__{family}'],
            family,
            metric_columns=STABILITY_SCORE_COLUMNS,
        ))
    family_stability = pd.concat(family_stability_tables, ignore_index=True)
    family_stability.to_csv(export_dir / 'relative_stability_score_comparison.csv', index=False)

    incremental_tables = []
    for group_key, spec in incremental_specs.items():
        baseline_path = path_by_name.get(spec['old_name'])
        candidate_path = path_by_name.get(spec['candidate_name'])
        if baseline_path is None or candidate_path is None:
            raise KeyError(f'Résultat incrémental incomplet pour {group_key}.')
        table = comparison_table(
            metrics,
            baseline_path,
            candidate_path,
            spec['family'],
            upgrade_label='candidate',
        )
        stability_table = comparison_table(
            relative_stability_scores,
            baseline_path,
            candidate_path,
            spec['family'],
            upgrade_label='candidate',
            metric_columns=STABILITY_SCORE_COLUMNS,
        )
        merge_keys = [column for column in ('scope', 'period_id', 'period_label') if column in table.columns]
        stability_columns = merge_keys + [
            'relative_smoothness_score_old', 'relative_smoothness_score_candidate',
            'delta_relative_smoothness_score',
            'relative_persistence_score_old', 'relative_persistence_score_candidate',
            'delta_relative_persistence_score',
        ]
        table = table.merge(
            stability_table.loc[:, [column for column in stability_columns if column in stability_table.columns]],
            on=merge_keys,
            how='left',
        )
        table.insert(1, 'variable', spec['variable'])
        table.insert(2, 'dimension', spec['dimension'])
        table.insert(3, 'weight', spec.get('weight', 1.0))
        table.insert(4, 'incremental_group', group_key)
        incremental_tables.append(table)
    incremental_effects = (
        pd.concat(incremental_tables, ignore_index=True) if incremental_tables else pd.DataFrame()
    )
    incremental_effects.to_csv(export_dir / 'incremental_effects.csv', index=False)
    if not incremental_effects.empty:
        incremental_effects.loc[incremental_effects['period_id'].eq('total')].to_csv(
            export_dir / 'incremental_effects_total.csv', index=False
        )

    manifest_rows = []
    for family, specs in composite_specs.items():
        manifest_rows.append({
            'market': market, 'family': family, 'role': 'old_factor',
            'variable': baseline_columns[family], 'dimension': 'level', 'weight': 1.0,
        })
        for index, spec in enumerate(specs, start=1):
            manifest_rows.append({
                'market': market, 'family': family, 'role': spec.get('role', f'component_{index}'),
                'variable': spec['variable'], 'dimension': spec['dimension'],
                'weight': spec.get('weight', 1.0),
                'higher_is_better': spec.get('higher_is_better'),
            })
    pd.DataFrame(manifest_rows).to_csv(export_dir / 'composite_config_manifest.csv', index=False)

    return {
        'market': market,
        'benchmark': benchmark,
        'baseline_columns': baseline_columns,
        'composite_specs': composite_specs,
        'incremental_specs': incremental_specs,
        'results': all_results,
        'export_dir': export_dir,
        'metrics': metrics,
        'registry': registry,
        'path_by_name': path_by_name,
        'family_comparison': family_comparison,
        'relative_stability_scores': relative_stability_scores,
        'family_stability': family_stability,
        'persistence_summary': persistence_summary,
        'incremental_effects': incremental_effects,
        'period_breakpoints': PERIOD_BREAKPOINTS,
    }


def plot_market_comparisons(bundle):
    """Crée une figure interactive par famille, ancien facteur contre upgrade."""
    figures_dir = bundle['export_dir'] / 'figures'
    data_dir = bundle['export_dir'] / 'data'
    figures_dir.mkdir(parents=True, exist_ok=True)
    data_dir.mkdir(parents=True, exist_ok=True)
    figures = {}
    for family in bundle['composite_specs']:
        old_path = bundle['path_by_name'][f'old__{family}']
        upgrade_path = bundle['path_by_name'][f'upgrade__{family}']
        selections = {
            f'Ancien | {family}': (old_path, 'Top'),
            f'Upgrade | {family}': (upgrade_path, 'Top'),
            'Benchmark': (old_path, 'Bench'),
        }
        performance, composition = combine_backtest_performances(
            results=bundle['results'],
            export_dir=bundle['export_dir'],
            selections=selections,
            return_composition=True,
        )
        ratios = calculate_performance_ratios(performance, benchmark_column='Benchmark')
        performance.to_csv(data_dir / f'{family}_old_vs_upgrade_performance.csv')
        composition.to_csv(data_dir / f'{family}_old_vs_upgrade_composition.csv', index=False)
        figure = plot_performance_comparison(
            performance=performance,
            ratios=ratios,
            benchmark_column='Benchmark',
            title=f"{bundle['market']} | {family} : ancien contre upgrade",
            save_path=figures_dir / f'{family}_old_vs_upgrade.html',
            show_plot=False,
            rebase=True,
            period_breakpoints=bundle['period_breakpoints'],
            default_period_id='total',
        )
        figures[family] = figure
        display(figure)
    print(f"Figures enregistrées dans : {figures_dir}")
    return figures


## 1. STOXX EUROPE 600

Le facteur historique est recherché en priorité sous la forme `Avg Percentile`, qui correspond aux résultats historiques collés dans la recherche. Chaque configuration contient volontairement le niveau de l'ancien facteur **et une dimension change de ce même facteur**, puis une ou deux variables satellites. Pour Quality, `FCF Conversion` et `Net Debt to Tot Equity` sont inclus explicitement comme signaux de conversion de cash et de levier, avec leur niveau et une dimension de changement. Les composantes ci-dessous sont des points de départ issus du rapport d'évidence et de tes essais manuels ; elles ne sont pas figées. Modifiez uniquement la liste de composantes et leurs poids avant d'exécuter la cellule suivante.

Chaque ligne de `COMPOSITE_CONFIGS_STOXX` contient `variable`, `dimension`, `weight` et `higher_is_better`. Les lignes `role='old_change'` sont les changements du facteur historique ; elles ne sont pas traitées comme des variables externes.

In [ ]:
# CELLULE À AJUSTER MANUELLEMENT : configuration STOXX.
STOXX_BASELINE_CANDIDATES = {
    'growth': ('Growth Avg Percentile', 'GROWTH_SCORE_FS_SECTOR'),
    'quality': ('Quality Avg Percentile', 'MARGIN_SCORE_FS_SECTOR'),
    'momentum': ('Mom Avg Percentile', 'MOMENTUM_SCORE_FS_SECTOR'),
    'value': ('Value Avg Percentile', 'VALUE_SCORE_FS_SECTOR'),
    'dividend': ('Dividend Avg Percentile', 'Dividend_NTM Avg Percentile'),
}
STOXX_AVAILABLE = set(pq.ParquetFile(SCREEN_PATH).schema_arrow.names)
STOXX_BASELINE_COLUMNS = resolve_baseline_columns(STOXX_BASELINE_CANDIDATES, STOXX_AVAILABLE)
STOXX_OLD = STOXX_BASELINE_COLUMNS

COMPOSITE_CONFIGS_STOXX = {
    'quality': [
        {'role': 'old_core', 'variable': STOXX_OLD['quality'], 'dimension': 'level', 'weight': 0.40, 'higher_is_better': True},
        {'role': 'old_change', 'variable': STOXX_OLD['quality'], 'dimension': 'diff_3', 'weight': 0.15, 'higher_is_better': True},
        {'role': 'cash_quality', 'variable': 'FCF Conversion', 'dimension': 'level', 'weight': 0.15, 'higher_is_better': True},
        {'role': 'cash_quality_change', 'variable': 'FCF Conversion', 'dimension': 'diff_3', 'weight': 0.10, 'higher_is_better': True},
        {'role': 'leverage_quality', 'variable': 'Net Debt to Tot Equity', 'dimension': 'level', 'weight': 0.05, 'higher_is_better': False},
        {'role': 'leverage_quality_change', 'variable': 'Net Debt to Tot Equity', 'dimension': 'diff_1', 'weight': 0.10, 'higher_is_better': False},
        {'role': 'leverage_confirmation', 'variable': 'NetDebt to EBITDA exFIN', 'dimension': 'rank_diff_3', 'weight': 0.05, 'higher_is_better': False},
    ],
    'growth': [
        {'role': 'old_core', 'variable': STOXX_OLD['growth'], 'dimension': 'level', 'weight': 0.60, 'higher_is_better': True},
        {'role': 'old_change', 'variable': STOXX_OLD['growth'], 'dimension': 'diff_3', 'weight': 0.20, 'higher_is_better': True},
        {'role': 'satellite', 'variable': 'CFO 5Y CAGR', 'dimension': 'level', 'weight': 0.20, 'higher_is_better': True},
    ],
    'momentum': [
        {'role': 'old_core', 'variable': STOXX_OLD['momentum'], 'dimension': 'level', 'weight': 0.60, 'higher_is_better': True},
        {'role': 'old_change', 'variable': STOXX_OLD['momentum'], 'dimension': 'pct_3', 'weight': 0.20, 'higher_is_better': True},
        {'role': 'satellite', 'variable': 'SP Price Target CIQ', 'dimension': 'pct_12', 'weight': 0.20, 'higher_is_better': True},
    ],
    'dividend': [
        {'role': 'old_core', 'variable': STOXX_OLD['dividend'], 'dimension': 'level', 'weight': 0.50, 'higher_is_better': True},
        {'role': 'old_change', 'variable': STOXX_OLD['dividend'], 'dimension': 'diff_3', 'weight': 0.20, 'higher_is_better': True},
        {'role': 'satellite', 'variable': 'CFO Div Cov Ratio', 'dimension': 'diff_3', 'weight': 0.15, 'higher_is_better': True},
        {'role': 'satellite', 'variable': 'DPS FY1', 'dimension': 'pct_6', 'weight': 0.15, 'higher_is_better': True},
    ],
    'value': [
        {'role': 'old_core', 'variable': STOXX_OLD['value'], 'dimension': 'level', 'weight': 0.55, 'higher_is_better': True},
        {'role': 'old_change', 'variable': STOXX_OLD['value'], 'dimension': 'diff_3', 'weight': 0.20, 'higher_is_better': True},
        {'role': 'satellite', 'variable': 'Earns Yield FY1', 'dimension': 'diff_6', 'weight': 0.15, 'higher_is_better': True},
        {'role': 'satellite', 'variable': 'EV To EBITDA LTM', 'dimension': 'pct_3', 'weight': 0.10, 'higher_is_better': False},
    ],
}

display(pd.DataFrame(
    [
        {'family': family, **spec}
        for family, specs in COMPOSITE_CONFIGS_STOXX.items()
        for spec in specs
    ]
).loc[:, ['family', 'role', 'variable', 'dimension', 'weight', 'higher_is_better']])

STOXX_BUNDLE = run_market(
    market='STOXX EUROPE 600',
    benchmark='STOXX EUROPE 600',
    output_name='factor_core_upgrade_comparison_STOXX600',
    baseline_candidates=STOXX_BASELINE_CANDIDATES,
    composite_specs=COMPOSITE_CONFIGS_STOXX,
)

display(display_key_comparison(STOXX_BUNDLE['family_comparison'].loc[STOXX_BUNDLE['family_comparison']['period_id'].eq('total')]))
display(display_key_stability(STOXX_BUNDLE['family_stability'].loc[STOXX_BUNDLE['family_stability']['period_id'].eq('total')]))
display(display_key_incremental(STOXX_BUNDLE['incremental_effects'].loc[STOXX_BUNDLE['incremental_effects']['period_id'].eq('total')]))
plot_market_comparisons(STOXX_BUNDLE)


### Lecture des sorties STOXX

- `family_composite_vs_old.csv` contient la comparaison par période entre l'ancien facteur et l'upgrade. Les colonnes `delta_*` sont calculées comme `upgrade - old`.
- `incremental_effects.csv` mesure séparément chaque composante ajoutée au facteur historique, y compris les dimensions `diff_*`, `pct_*` ou `rank_diff_*` du même ancien facteur. Il contient aussi `delta_relative_smoothness_score` et `delta_relative_persistence_score` pour juger l'effet sur la courbe relative. Une composante ne doit pas être conservée sur le seul CAGR : vérifier aussi `top_worst_cagr`, `top_information_ratio`, le drawdown actif et le tracking error.
- `relative_stability_scores.csv` ne conserve que deux scores sur 0-100 pour la courbe `Top / Benchmark` : `relative_smoothness_score` combine tendance, bruit relatif, creux relatifs et récupération ; `relative_persistence_score` combine les fenêtres positives et la fréquence de surperformance.
- `relative_stability_score_comparison.csv` compare ces deux scores entre ancien facteur et upgrade, avec leurs deltas par période.
- `figures/<family>_old_vs_upgrade.html` contient la courbe Top de l'ancien facteur, la courbe Top de l'upgrade et le benchmark, avec le sélecteur de période.

## 2. EUROPE SMALL CAP

Cette section est volontairement indépendante de STOXX : elle recharge le même screen, utilise le benchmark `MSCI EUR SMALL`, mais possède ses propres facteurs historiques et sa propre configuration composite. Comme dans la section STOXX, les lignes `old_change` permettent d'utiliser les changements utiles des anciens facteurs. Les poids et les variables peuvent être changés sans toucher à la section STOXX.

In [ ]:
# CELLULE À AJUSTER MANUELLEMENT : configuration EUROPE SMALL CAP.
SMALL_BASELINE_CANDIDATES = {
    'growth': ('Growth Avg Percentile', 'GROWTH_SCORE_FS_SECTOR'),
    'quality': ('Quality Avg Percentile', 'MARGIN_SCORE_FS_SECTOR'),
    'momentum': ('Mom Avg Percentile', 'MOMENTUM_SCORE_FS_SECTOR'),
    'value': ('Value Avg Percentile', 'VALUE_SCORE_FS_SECTOR'),
    'dividend': ('Dividend Avg Percentile', 'Dividend_NTM Avg Percentile'),
}
SMALL_AVAILABLE = set(pq.ParquetFile(SCREEN_PATH).schema_arrow.names)
SMALL_BASELINE_COLUMNS = resolve_baseline_columns(SMALL_BASELINE_CANDIDATES, SMALL_AVAILABLE)
SMALL_OLD = SMALL_BASELINE_COLUMNS

COMPOSITE_CONFIGS_SMALL = {
    'quality': [
        {'role': 'old_core', 'variable': SMALL_OLD['quality'], 'dimension': 'level', 'weight': 0.60, 'higher_is_better': True},
        {'role': 'old_change', 'variable': SMALL_OLD['quality'], 'dimension': 'diff_3', 'weight': 0.20, 'higher_is_better': True},
        {'role': 'satellite', 'variable': 'PCT ROE', 'dimension': 'diff_3', 'weight': 0.20, 'higher_is_better': True},
    ],
    'growth': [
        {'role': 'old_core', 'variable': SMALL_OLD['growth'], 'dimension': 'level', 'weight': 0.60, 'higher_is_better': True},
        {'role': 'old_change', 'variable': SMALL_OLD['growth'], 'dimension': 'rank_diff_6', 'weight': 0.20, 'higher_is_better': True},
        {'role': 'satellite', 'variable': 'PCT Hist GrossInc', 'dimension': 'rank_diff_3', 'weight': 0.20, 'higher_is_better': True},
    ],
    'momentum': [
        {'role': 'old_core', 'variable': SMALL_OLD['momentum'], 'dimension': 'level', 'weight': 0.60, 'higher_is_better': True},
        {'role': 'old_change', 'variable': SMALL_OLD['momentum'], 'dimension': 'pct_6', 'weight': 0.20, 'higher_is_better': True},
        {'role': 'satellite', 'variable': 'SP Price Target CIQ', 'dimension': 'pct_6', 'weight': 0.20, 'higher_is_better': True},
    ],
    'dividend': [
        {'role': 'old_core', 'variable': SMALL_OLD['dividend'], 'dimension': 'level', 'weight': 0.60, 'higher_is_better': True},
        {'role': 'old_change', 'variable': SMALL_OLD['dividend'], 'dimension': 'diff_3', 'weight': 0.20, 'higher_is_better': True},
        {'role': 'satellite', 'variable': 'PCT DvdYield FY1', 'dimension': 'level', 'weight': 0.20, 'higher_is_better': True},
    ],
    'value': [
        {'role': 'old_core', 'variable': SMALL_OLD['value'], 'dimension': 'level', 'weight': 0.55, 'higher_is_better': True},
        {'role': 'old_change', 'variable': SMALL_OLD['value'], 'dimension': 'diff_3', 'weight': 0.20, 'higher_is_better': True},
        {'role': 'satellite', 'variable': 'Earns Yield FY0', 'dimension': 'level', 'weight': 0.15, 'higher_is_better': True},
        {'role': 'satellite', 'variable': 'EV To EBITDA LTM', 'dimension': 'rank_diff_3', 'weight': 0.10, 'higher_is_better': False},
    ],
}

display(pd.DataFrame(
    [
        {'family': family, **spec}
        for family, specs in COMPOSITE_CONFIGS_SMALL.items()
        for spec in specs
    ]
).loc[:, ['family', 'role', 'variable', 'dimension', 'weight', 'higher_is_better']])

SMALL_BUNDLE = run_market(
    market='EUROPE SMALL CAP',
    benchmark='MSCI EUR SMALL',
    output_name='factor_core_upgrade_comparison_SMALL',
    baseline_candidates=SMALL_BASELINE_CANDIDATES,
    composite_specs=COMPOSITE_CONFIGS_SMALL,
)

display(display_key_comparison(SMALL_BUNDLE['family_comparison'].loc[SMALL_BUNDLE['family_comparison']['period_id'].eq('total')]))
display(display_key_stability(SMALL_BUNDLE['family_stability'].loc[SMALL_BUNDLE['family_stability']['period_id'].eq('total')]))
display(display_key_incremental(SMALL_BUNDLE['incremental_effects'].loc[SMALL_BUNDLE['incremental_effects']['period_id'].eq('total')]))
plot_market_comparisons(SMALL_BUNDLE)


### Lecture finale

Commencer par `family_composite_vs_old_total.csv`, puis lire `relative_stability_score_comparison.csv` et `incremental_effects_total.csv`. Un score de lissage ou de persistance plus élevé est préférable, mais il ne remplace pas le contrôle de l'alpha : conserver aussi `active_cagr`, `top_information_ratio` et `top_worst_cagr` positifs, avec un risque relatif non dégradé.

Les lignes `before_2009` ou les périodes sans observations doivent rester traitées comme absentes, et non comme une performance nulle. Le notebook ne promeut aucun facteur en production : il produit uniquement les tables et figures nécessaires à la revue de recherche.